In [ ]:
import numpy as np
import torch
torch.set_default_dtype(torch.float32)
import gpytorch

import gc

dtype = torch.float32
device = 'cuda:0'

In [ ]:
def free():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
y_train = np.load('../data/y_train.npy')
n = y_train.shape[0]

n_samples = n
idx = np.random.choice(range(n), size=n_samples, replace=False)
idx = range(n)


X_train = torch.tensor(np.load('../data/X_train.npy')[idx], dtype=dtype, device=device)
y_train = torch.tensor(y_train[idx], dtype=dtype, device=device)
y_mean = y_train.mean()
y_std = y_train.std()

# y_train = (y_train - y_mean) / y_std

# Training

In [ ]:
import torch
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy

# ── 1. Mean Function ──────────────────────────────────────────────────────────

class IndoorOutdoorMean(gpytorch.means.Mean):
    """Learns a separate constant mean for indoor (group=0) and outdoor (group=1)."""
    def __init__(self):
        super().__init__()
        # One learnable mean per group
        #self.register_parameter(
        #    "group_means", torch.nn.Parameter(torch.zeros(2))
        #)
        group_idx = X_train[:,2].long()
        self.group_means = torch.tensor([y_train[~group_idx].mean(), y_train[group_idx].mean()],
                                       dtype=dtype, device=device)

    def forward(self, x):
        # x: [N, 3] — columns are [lon, lat, group]
        group_idx = x[:, 2].long()          # integer group label
        return self.group_means[group_idx]  # [N]


# ── 2. Per-Group Noise Likelihood ─────────────────────────────────────────────

class HeteroscedasticGroupLikelihood(gpytorch.likelihoods.Likelihood):
    def __init__(self):
        super().__init__()
        self.register_parameter(
            "raw_noise", torch.nn.Parameter(torch.zeros(2))
        )

    @property
    def noise(self):
        return torch.nn.functional.softplus(self.raw_noise)

    def forward(self, function_samples, x=None, **kwargs):
        # Pop x before anything gets forwarded to the distribution
        group_idx = x[:, 2].long()
        noise_std = self.noise[group_idx].sqrt()
        return torch.distributions.Normal(function_samples, noise_std)

    def expected_log_prob(self, observations, function_dist, x=None, **kwargs):
        """Called by VariationalELBO during training."""
        group_idx = x[:, 2].long()
        noise_var = self.noise[group_idx]

        # E[log p(y | f)] for Gaussian = -0.5 * (log(2π σ²) + (y - μ)²/σ² + Var[f]/σ²)
        mean = function_dist.mean
        variance = function_dist.variance

        res = -0.5 * (
            noise_var.log()
            + (observations - mean).pow(2) / noise_var
            + variance / noise_var
            + torch.log(torch.tensor(2 * torch.pi))
        )
        return res.sum(-1)

    def marginal(self, function_dist, x=None, **kwargs):
        """Called at prediction time."""
        group_idx = x[:, 2].long()
        noise_var = self.noise[group_idx]

        mean = function_dist.mean
        var  = function_dist.variance + noise_var
        return gpytorch.distributions.MultivariateNormal(
            mean, torch.diag_embed(var)
        )


# ── 3. Sparse GP Model (handles dense/clustered data) ────────────────────────

class SparseRSSIModel(ApproximateGP):
    def __init__(self, inducing_points):
        """
        inducing_points: [M, 3] — (lon, lat, group), same shape as train_x.
        """
        variational_distribution = CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = VariationalStrategy(
            self,
            inducing_points,          # must be [M, 3] to match train_x
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)

        self.mean_module = IndoorOutdoorMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=2)
        )

    def forward(self, x):
        # x is [N, 3]; slice here for the kernel only
        x_spatial = x[:, :2]

        mean_x  = self.mean_module(x)
        covar_x = self.covar_module(x_spatial)

        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


# ── 4. Setup & Training ───────────────────────────────────────────────────────

def train(train_x, train_y, n_epochs=200, lr=0.01, n_inducing=200):
    """
    train_x: [N, 3]  columns = [lon_norm, lat_norm, group]  (group: 0=indoor, 1=outdoor)
    train_y: [N]     RSSI in dBm
    """
    # Initialise inducing points from a subset of spatial training locations
    indices = torch.randperm(train_x.size(0))[:n_inducing]
    inducing_points = train_x[indices].clone()  # lon/lat only

    model      = SparseRSSIModel(inducing_points).to(train_x.device)
    likelihood = HeteroscedasticGroupLikelihood().to(train_x.device)

    model.covar_module.outputscale = torch.tensor(
        train_y.var().item()/2, device=train_x.device)  # prior variance in dBm²
    target_noise = torch.tensor([train_y.var().item()/2]*2, device=train_x.device)   # indoor σ²=10, outdoor σ²=5
    # ── Per-group noise (via inverse softplus, since raw_noise feeds into softplus)
    def inv_softplus(x):
        return x + torch.log(-torch.expm1(-x))  # numerically stable inverse
    likelihood.raw_noise.data = inv_softplus(target_noise)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(likelihood.parameters()), lr=lr
    )

    # ELBO loss for approximate (sparse) GPs
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_x.size(0))

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss   = -mll(output, train_y, x=train_x)   # pass x for group-aware noise
        loss.backward()
        optimizer.step()

        if epoch % (n_epochs // 10) == 0:
            noise = likelihood.noise
            print(
                f"Epoch {epoch:3d} | Loss: {loss.item():.3f} | "
                f"σ²_indoor: {noise[0].item():.4f} | "
                f"σ²_outdoor: {noise[1].item():.4f}"
            )
            lengthscales = model.covar_module.base_kernel.lengthscale[0]
            print(
                "           "
                f"Lengthscales: [{lengthscales[0]:.4f}, {lengthscales[1]:.4f}]"
            )
            print(
                "           "
                f"Outputscale: {model.covar_module.outputscale:.4f}"
            )

    return model, likelihood


# ── 5. Prediction ─────────────────────────────────────────────────────────────

def predict(model, likelihood, test_x):
    """test_x: [N, 3]  same column order as training."""
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = likelihood(model(test_x), x=test_x)
        mean  = preds.mean
        lower, upper = preds.confidence_region()  # ±2σ

    return mean, lower, upper

In [ ]:
n_inducing = int(np.sqrt(n))//2
inducing_idx = np.random.choice(range(n), size=n_inducing, replace=False)
inducing_pts = X_train[inducing_idx, :2]

In [ ]:
model, likelihood = train(X_train, y_train, n_inducing=n_inducing, lr=0.01, n_epochs=10_000)

In [ ]:
X_test = torch.tensor(np.load('../data/X_test.npy'), dtype=dtype, device=device)
y_hat, lower_ci, upper_ci = predict(model, likelihood, X_test)

In [ ]:
import sys
import os
sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")


from wifiplotting import *

import dill

with open('../data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)
wlon_train, wlat_train = np.load('../data/world_train.npy').T
wlon_test, wlat_test = np.load('../data/world_test.npy').T

# Plotting


In [ ]:
with torch.no_grad():
    vmax = torch.quantile(y_train, 0.975)
    vmin = torch.quantile(y_train, 0.025)
    
    base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)
    
    sc = base_ax.scatter(wlon_test, wlat_test, c=upper_ci.cpu().numpy(), cmap="RdYlGn", vmin=vmin, vmax=vmax)
    
    tr = base_ax.scatter(wlon_train[idx], wlat_train[idx], c=y_train.cpu().numpy(), cmap="RdYlGn",
                    vmin=vmin, vmax=vmax, alpha=1)
    plt.ticklabel_format(style='plain', axis='both', useOffset=False)
    
    plt.colorbar(tr)
    plt.tight_layout()

In [ ]:
model.mean_module.group_means
